In [1]:

import pandas as pd

csv_path = "entropy-wave-convergence-2d.csv"
tex_path = "entropy-wave-convergence-2d.tex"


In [2]:
# Keep the same method order as the benchmark.
configs = [
    "MUSCL-Hancock",
    "MUSCL-RK3",
    "ZS3",
    "ZS3lazy",
    "MM3/1rev/rtol_1e-1",
    "MM3/1rev/rtol_0",
    "ZS7",
    "ZS7lazy",
    "MM7/1rev/rtol_1e-1",
    "MM7/1rev/rtol_0",
]

labels = {
    "MUSCL-Hancock": "MUSCL-Hancock",
    "MUSCL-RK3": r"MUSCL$+$SSPRK3",
    "ZS3": r"4th-order \textit{a priori}",
    "ZS3lazy": r"4th-order \textit{a priori}, $\overline{\boldsymbol{W}}^{(1)}$",
    "MM3/1rev/rtol_1e-1": r"4th-order \textit{a posteriori}, $\epsilon=10^{-1}$",
    "MM3/1rev/rtol_0": r"4th-order \textit{a posteriori}, $\epsilon=0$",
    "ZS7": r"8th-order \textit{a priori}",
    "ZS7lazy": r"8th-order \textit{a priori}, $\overline{\boldsymbol{W}}^{(1)}$",
    "MM7/1rev/rtol_1e-1": r"8th-order \textit{a posteriori}, $\epsilon=10^{-1}$",
    "MM7/1rev/rtol_0": r"8th-order \textit{a posteriori}, $\epsilon=0$",
}

df = pd.read_csv(csv_path)
df["name"] = pd.Categorical(df["name"], categories=configs, ordered=True)
df = df.sort_values(["name", "N"])


In [3]:
# Inspect the recorded values before formatting for the manuscript.
table = df.set_index(["name", "N"])[["L1", "L1_rate", "Linf", "Linf_rate"]]
table


L1   L1_rate          Linf  Linf_rate
name               N                                                   
MUSCL-Hancock      32   4.142734e-03       NaN  8.416701e-03        NaN
                   64   5.933682e-04  2.803584  1.576243e-03   2.416764
                   128  1.320659e-04  2.167670  3.166879e-04   2.315357
MUSCL-RK3          32   2.618472e-02       NaN  4.154440e-02        NaN
                   64   6.458106e-03  2.019542  1.020776e-02   2.024988
                   128  1.608470e-03  2.005422  2.531738e-03   2.011467
ZS3                32   2.321833e-04       NaN  5.149037e-04        NaN
                   64   1.082051e-05  4.423424  2.570554e-05   4.324151
                   128  5.860906e-07  4.206501  1.453661e-06   4.144316
ZS3lazy            32   4.246721e-03       NaN  1.131524e-02        NaN
                   64   1.252438e-03  1.761611  3.232554e-03   1.807521
                   128  3.261789e-04  1.941003  8.347503e-04   1.953258
MM3/1rev/rtol_1e-1 32   2.471907e-04       NaN  5.168844e-04        NaN
                   64   1.212094e-05  4.350051  2.902871e-05   4.154289
                   128  6.665419e-07  4.184663  1.725287e-06   4.072572
MM3/1rev/rtol_0    32   4.754484e-02       NaN  1.023729e-01        NaN
                   64   1.415662e-02  1.747812  3.865367e-02   1.405156
                   128  3.848428e-03  1.879135  1.681063e-02   1.201231
ZS7                32   1.498161e-07       NaN  3.092567e-07        NaN
                   64   2.765167e-10  9.081611  6.116704e-10   8.981835
                   128  7.697465e-13  8.488767  1.889378e-12   8.338700
ZS7lazy            32   4.615431e-03       NaN  1.229213e-02        NaN
                   64   1.259338e-03  1.873800  3.203724e-03   1.939913
                   128  3.274797e-04  1.943188  8.377035e-04   1.935238
MM7/1rev/rtol_1e-1 32   1.745390e-07       NaN  3.356759e-07        NaN
                   64   4.245388e-10  8.683437  9.482631e-10   8.467566
                   128  1.085626e-12  8.611226  2.874256e-12   8.365955
MM7/1rev/rtol_0    32   4.098093e-02       NaN  9.490068e-02        NaN
                   64   9.668031e-03  2.083659  2.916669e-02   1.702097
                   128  1.925762e-03  2.327793  1.079415e-02   1.434072

In [4]:
def format_error(value):
    if pd.isna(value):
        return "--"
    mantissa, exponent = f"{value:.2e}".split("e")
    return rf"${mantissa} \times 10^{{{int(exponent)}}}$"


def format_rate(value):
    return "--" if pd.isna(value) else f"{value:.2f}"


# Rates are log(E_coarse / E_fine) / log(N_fine / N_coarse).
# The first resolution and zero-error comparisons have undefined rates.
df_to_table = df.copy()
df_to_table["Method"] = [labels[name] for name in df_to_table["name"]]
for norm in ("L1", "Linf"):
    df_to_table[norm] = df_to_table[norm].map(format_error)
    df_to_table[f"{norm}_rate"] = df_to_table[f"{norm}_rate"].map(format_rate)

manuscript_table = df_to_table.set_index(["Method", "N"])[
    ["L1", "L1_rate", "Linf", "Linf_rate"]
]
manuscript_table.columns = [r"$L^1$ error", "Rate", r"$L^\infty$ error", "Rate"]

# Requires \usepackage{booktabs, amsmath, amssymb} in the manuscript preamble.
latex = manuscript_table.to_latex(
    escape=False,
    column_format="lrrrrr",
    multirow=False,
    caption=(
        r"Two-dimensional entropy-wave convergence at $t=1$. Errors compare "
        r"$v_z$ with its initial state; $L^1$ is the mean absolute error. "
        r"Rates compare successive resolutions within each method."
    ),
    label="tab:entropy-wave-convergence-2d",
)
with open(tex_path, "w") as f:
    f.write(latex)
print(latex)


\begin{table}
\caption{Two-dimensional entropy-wave convergence at $t=1$. Errors compare $v_z$ with its initial state; $L^1$ is the mean absolute error. Rates compare successive resolutions within each method.}
\label{tab:entropy-wave-convergence-2d}
\begin{tabular}{lrrrrr}
\toprule
 &  & $L^1$ error & Rate & $L^\infty$ error & Rate \\
Method & N &  &  &  &  \\
\midrule
MUSCL-Hancock & 32 & $4.14 \times 10^{-3}$ & -- & $8.42 \times 10^{-3}$ & -- \\
 & 64 & $5.93 \times 10^{-4}$ & 2.80 & $1.58 \times 10^{-3}$ & 2.42 \\
 & 128 & $1.32 \times 10^{-4}$ & 2.17 & $3.17 \times 10^{-4}$ & 2.32 \\
MUSCL$+$SSPRK3 & 32 & $2.62 \times 10^{-2}$ & -- & $4.15 \times 10^{-2}$ & -- \\
 & 64 & $6.46 \times 10^{-3}$ & 2.02 & $1.02 \times 10^{-2}$ & 2.02 \\
 & 128 & $1.61 \times 10^{-3}$ & 2.01 & $2.53 \times 10^{-3}$ & 2.01 \\
4th-order \textit{a priori} & 32 & $2.32 \times 10^{-4}$ & -- & $5.15 \times 10^{-4}$ & -- \\
 & 64 & $1.08 \times 10^{-5}$ & 4.42 & $2.57 \times 10^{-5}$ & 4.32 \\
 & 128 & $5.86 